# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---


In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip uninstall -y accelerate transformers
!pip install -U safetensors
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00


In [ ]:
ls ../input/competitions/pixels-to-predictions/

images/  sample_submission.csv  test.csv  train.csv  val.csv


In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("../input/competitions/pixels-to-predictions/")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Load and Preprocess Data


In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
  df["choices"] = df["choices"].apply(json.loads)

print(
    f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"


def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
  """
  Builds the text prompt for the Vision Language Model.
  The <image> token is required for the model to process the image.
  """
  context_parts = []
  lecture = row.get("lecture", "")
  hint = row.get("hint", "")
  if pd.notna(lecture) and str(lecture).strip():
    context_parts.append(str(lecture).strip())
  if pd.notna(hint) and str(hint).strip():
    context_parts.append(str(hint).strip())
  context_str = "\n".join(context_parts)

  choices = row["choices"]
  choices_str = "\n".join(
      f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
  )

  prompt = "<image>\n"
  if context_str:
    prompt += f"Context:\n{context_str}\n\n"
  prompt += f"Question: {row['question']}\n"
  prompt += f"Choices:\n{choices_str}\n"
  prompt += "Answer:"

  if include_answer:
    answer_idx = int(row['answer'])
    prompt += f" {CHOICE_LETTERS[answer_idx]}"

  return prompt


# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
  def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
    self.df = df.reset_index(drop=True)
    self.data_dir = data_dir
    self.img_size = img_size
    self.is_train = is_train

  def __len__(self) -> int:
    return len(self.df)

  def _load_image(self, rel_path: str) -> Image.Image:
    img = Image.open(self.data_dir / rel_path).convert("RGB")
    img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
    return img

  def __getitem__(self, idx: int) -> dict:
    row = self.df.iloc[idx]
    img = self._load_image(row["image_path"])

    if self.is_train:
      return {
          "image":  img,
          "text":   build_prompt(row, include_answer=True),
          "answer": int(row["answer"]),
      }
    else:
      return {
          "image":   img,
          "text":    build_prompt(row, include_answer=False),
          "choices": row["choices"],
          "answer":  int(row["answer"]) if "answer" in row else -1,
      }


train_ds = ScienceQADataset(
    train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds = ScienceQADataset(
    val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds = ScienceQADataset(
    test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(
    f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.


In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
  processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
  model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(
    DATA_DIR / "images" / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v)
          else v for k, v in inputs.items()}

with torch.inference_mode():
  generated_ids = model.generate(
      **inputs,
      max_new_tokens=20,
      do_sample=False,
  )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

## 4. Strict Multiple-Choice Evaluator

This section evaluates answer choices by scoring only valid choice letters (A..E as needed) at the `Answer:` position. It avoids free-form generation errors and gives a stable validation metric.


In [ ]:
# ── 4a. Strict evaluator utilities ─────────────────────────────────────────
from pathlib import Path
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

CHOICE_LETTERS = "ABCDEFGHIJ"

OUTPUT_DIR = Path(
    "/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")


def resolve_image_path(data_dir: Path, rel_path: str) -> Path:
  rel = Path(rel_path)
  stripped = Path(
      *rel.parts[1:]) if len(rel.parts) > 1 and rel.parts[0] == "images" else rel
  candidates = [
      data_dir / rel,
      data_dir / stripped,
      data_dir / "images" / rel,
      data_dir / "images" / stripped,
  ]
  for p in candidates:
    if p.exists():
      return p
  raise FileNotFoundError(
      f"Could not resolve image path for {rel_path}. Tried: {candidates}")


def get_choice_token_id(letter: str) -> int:
  # Prefer single-token forms; try with and without leading space
  variants = [f" {letter}", letter]
  for txt in variants:
    ids = processor.tokenizer(txt, add_special_tokens=False).input_ids
    if len(ids) == 1:
      return ids[0]
  raise ValueError(f"Could not map choice letter {letter} to one token.")


CHOICE_TOKEN_IDS = {ch: get_choice_token_id(ch) for ch in CHOICE_LETTERS}
print("Choice token ids:", {k: CHOICE_TOKEN_IDS[k] for k in "ABCDE"})


def score_choices_for_row(row: pd.Series):
  n = int(row["num_choices"])
  valid_letters = CHOICE_LETTERS[:n]
  image = Image.open(resolve_image_path(DATA_DIR, row["image_path"])).convert(
      "RGB").resize((IMG_SIZE, IMG_SIZE))

  if "build_prompt_cfg" in globals():
    prompt_base = build_prompt_cfg(row, include_answer=False)
  else:
    prompt_base = build_prompt(row, include_answer=False)

  full_texts = [prompt_base + f" {valid_letters[i]}" for i in range(n)]
  processor.tokenizer.padding_side = "right"
  inputs = processor(text=full_texts, images=[image] * n, return_tensors="pt", padding=True)
  inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v)
            for k, v in inputs.items()}

  eos_id = processor.tokenizer.eos_token_id
  pad_id = processor.tokenizer.pad_token_id

  with torch.inference_mode():
    logits = model(**inputs).logits

  log_probs = []
  for i in range(n):
    seq = inputs["input_ids"][i].tolist()
    answer_pos = len(seq) - 1
    while answer_pos >= 0 and seq[answer_pos] in (eos_id, pad_id):
      answer_pos -= 1
    pred_pos = max(answer_pos - 1, 0)
    answer_token_id = seq[answer_pos]
    lp = torch.log_softmax(logits[i, pred_pos], dim=-1)[answer_token_id].item()
    log_probs.append(lp)

  scores_t = torch.tensor(log_probs, dtype=torch.float32)
  valid_probs = F.softmax(scores_t, dim=-1)
  pred_idx = int(torch.argmax(scores_t).item())
  return pred_idx, valid_letters, valid_probs.detach().cpu().numpy()


In [ ]:
if not globals().get('RUN_BASELINE_SECTION4', False):
    print('Skipping section 4b baseline loop (RUN_BASELINE_SECTION4=False).')
else:
    # ── 4b. Strict validation evaluation ───────────────────────────────────────
    model.eval()

    val_rows = []
    correct = 0

    for i in tqdm(range(len(val_df)), desc="Strict val eval"):
      row = val_df.iloc[i]
      pred_idx, valid_letters, valid_probs = score_choices_for_row(row)
      gt_idx = int(row["answer"])
      is_correct = int(pred_idx == gt_idx)
      correct += is_correct

      rec = {
          "id": row["id"],
          "num_choices": int(row["num_choices"]),
          "answer": gt_idx,
          "pred": pred_idx,
          "correct": is_correct,
      }
      for j, letter in enumerate(valid_letters):
        rec[f"p_{letter}"] = float(valid_probs[j])
      val_rows.append(rec)

    val_pred_df = pd.DataFrame(val_rows)
    val_acc = correct / len(val_df)
    print(
        f"Strict evaluator validation accuracy: {val_acc:.4f} ({correct}/{len(val_df)})")

    val_out = OUTPUT_DIR / "val_strict_eval_predictions.csv"
    val_pred_df.to_csv(val_out, index=False)
    print(f"Saved: {val_out}")
    val_pred_df.head()


In [ ]:
if not globals().get('RUN_BASELINE_SECTION4', False):
    print('Skipping section 4c baseline loop (RUN_BASELINE_SECTION4=False).')
else:
    # ── 4c. Strict test inference + Kaggle submission ─────────────────────────
    test_preds = []

    for i in tqdm(range(len(test_df)), desc="Strict test infer"):
      row = test_df.iloc[i]
      pred_idx, _, _ = score_choices_for_row(row)
      test_preds.append(pred_idx)

    submission = pd.DataFrame({"id": test_df["id"], "answer": test_preds})
    sub_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(sub_path, index=False)
    print(f"Saved Kaggle submission: {sub_path}")
    submission.head()


## 5. QLoRA Fine-Tuning (Validation-First Screening)

This section is config-driven and optimized for screening runs: train with QLoRA, evaluate on validation, and only run test inference for promoted configs.


In [ ]:
# ── 5a. Experiment config (edit per run) ─────────────────────────────────
import time
from pathlib import Path

CFG = {
    "run_name": f"qlora-left-inspired-{time.strftime('%Y%m%d-%H%M%S')}",
    "seed": 42,
    "paths": {
        "data_dir": str(DATA_DIR),
        "output_dir": "/kaggle/working" if Path("/kaggle/working").exists() else "working",
        "pretrained_adapter_dir": "/kaggle/input/exp17-best-adapter/best_adapter",
    },
    "mode": {
        "do_train": True,
        "do_val_eval": False,
        "do_test_inference": True,
    },
    "model": {
        "model_id": MODEL_ID,
        "img_size": 160,
        "max_new_tokens": 4,
    },
    "prompting": {
        "use_lecture": True,
        "use_hint": True,
        "lecture_max_chars": 0,
        "hint_max_chars": 0,
    },
    "training": {
        "num_epochs": 1,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "warmup_ratio": 0.05,
        "max_grad_norm": 1.0,
        "logging_steps": 20,
    },
    "qlora": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "lora_r": 8,
        "lora_r_candidates": [12, 10, 8],
        "lora_alpha": 16,
        "lora_alpha_multiplier": 2,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "max_trainable_params": 5_000_000,
    },
    "data": {
        "augment_choices_train": True,
    },
    "screening": {
        "max_train_samples": None,
        "max_val_samples": 100,
        "verbose_every": 50,
    },
    "inference": {
        "scoring_mode": "letter",
        "hybrid_weight_fullchoice": 0.80,
    },
}

RUN_DIR = Path(CFG["paths"]["output_dir"]) / CFG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {RUN_DIR}")

RUN_BASELINE_SECTION4 = False
print(f"RUN_BASELINE_SECTION4={RUN_BASELINE_SECTION4}")


In [ ]:
# ── 5b. Config-aware prompt builder + answer-token-only collator ───────────
def _clip_text(x, max_chars):
    if x is None:
        return ""
    x = str(x).strip()
    if not x:
        return ""
    return x[:max_chars] if max_chars and len(x) > max_chars else x


def build_prompt_cfg(row: pd.Series, include_answer: bool = False) -> str:
    use_lecture = CFG["prompting"]["use_lecture"]
    use_hint = CFG["prompting"]["use_hint"]
    lecture_max_chars = CFG["prompting"]["lecture_max_chars"]
    hint_max_chars = CFG["prompting"]["hint_max_chars"]

    context_parts = []
    if use_lecture:
        lecture = _clip_text(row.get("lecture", ""), lecture_max_chars)
        if lecture:
            context_parts.append(lecture)
    if use_hint:
        hint = _clip_text(row.get("hint", ""), hint_max_chars)
        if hint:
            context_parts.append(hint)

    context_str = "\n".join(context_parts)
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"
    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"
    return prompt


class TrainScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].copy()

        if CFG.get("data", {}).get("augment_choices_train", False):
            choices = list(row["choices"])
            answer = int(row["answer"])
            perm = list(range(len(choices)))
            random.shuffle(perm)
            row["choices"] = [choices[p] for p in perm]
            row["answer"] = perm.index(answer)
        img = Image.open(resolve_image_path(DATA_DIR, row["image_path"]))                 .convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        answer_letter = CHOICE_LETTERS[int(row["answer"])]
        prompt_no_answer = build_prompt_cfg(row, include_answer=False)
        return {
            "image": img,
            "prompt": prompt_no_answer,
            "answer_letter": answer_letter,
        }


class TrainCollator:
    """Train on only the answer-token prediction after `Answer:`."""
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        prompts = [x["prompt"] for x in batch]
        answer_letters = [x["answer_letter"] for x in batch]
        images = [x["image"] for x in batch]

        # Full training text includes the gold answer token.
        full_texts = [f"{p} {a}" for p, a in zip(prompts, answer_letters)]
        out = self.processor(
            text=full_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

        labels = out["input_ids"].clone()
        labels[:] = -100

        # Supervise only the final non-pad token (= answer token).
        attn = out.get("attention_mask")
        if attn is None:
            raise ValueError("attention_mask is required for answer-only labeling")

        for i in range(labels.size(0)):
            last_idx = int(attn[i].sum().item()) - 1
            if last_idx < 0:
                continue
            labels[i, last_idx] = out["input_ids"][i, last_idx]

        out["labels"] = labels
        return out


In [ ]:
# ── 5c. Load 4-bit model and attach LoRA ─────────────────────────────────
import random
import numpy as np
import torch
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training


def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])

compute_dtype = torch.float16 if CFG["qlora"]["bnb_4bit_compute_dtype"] == "float16" else torch.bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CFG["qlora"]["load_in_4bit"],
    bnb_4bit_quant_type=CFG["qlora"]["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=compute_dtype,
)

processor = AutoProcessor.from_pretrained(CFG["model"]["model_id"])
if processor.tokenizer.pad_token is None:
  processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    CFG["model"]["model_id"],
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

if CFG["mode"]["do_train"]:
  model = prepare_model_for_kbit_training(model)

  # Auto-select near-cap LoRA rank safely (without repeated PEFT wrapping side effects).
  r_candidates = CFG["qlora"].get("lora_r_candidates", [CFG["qlora"].get("lora_r", 8)])
  r_candidates = sorted({int(r) for r in r_candidates})
  alpha_mult = int(CFG["qlora"].get("lora_alpha_multiplier", 2))
  max_cap = int(CFG["qlora"]["max_trainable_params"])

  r_ref = min(r_candidates)
  a_ref = alpha_mult * r_ref
  lora_cfg_ref = LoraConfig(
      r=r_ref,
      lora_alpha=a_ref,
      lora_dropout=CFG["qlora"]["lora_dropout"],
      target_modules=CFG["qlora"]["target_modules"],
      bias="none",
      task_type="CAUSAL_LM",
  )
  model = get_peft_model(model, lora_cfg_ref)
  trainable_ref = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  print(f"Reference LoRA r={r_ref}, alpha={a_ref}: {trainable_ref:,} / {total:,}")

  best_r = None
  for r_try in sorted(r_candidates, reverse=True):
    est = int(round(trainable_ref * (r_try / r_ref)))
    print(f"Estimated trainable for r={r_try}: {est:,}")
    if est <= max_cap and best_r is None:
      best_r = r_try

  if best_r is None:
    raise AssertionError(f"No cap-safe LoRA rank found under {max_cap:,}. Tried: {sorted(r_candidates, reverse=True)}")

  if best_r != r_ref:
    model = AutoModelForVision2Seq.from_pretrained(
        CFG["model"]["model_id"],
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    a_sel = alpha_mult * best_r
    lora_cfg_sel = LoraConfig(
        r=best_r,
        lora_alpha=a_sel,
        lora_dropout=CFG["qlora"]["lora_dropout"],
        target_modules=CFG["qlora"]["target_modules"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg_sel)
  else:
    a_sel = a_ref

  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  pct = 100.0 * trainable / total
  CFG["qlora"]["lora_r"] = int(best_r)
  CFG["qlora"]["lora_alpha"] = int(a_sel)
  print(f"Selected LoRA r={best_r}, alpha={a_sel}")
  assert trainable <= max_cap, f"Trainable params {trainable:,} exceed cap {max_cap:,}"
  print(f"Trainable params: {trainable:,} / {total:,} ({pct:.4f}%)")
else:
  adapter_dir = CFG["paths"].get("pretrained_adapter_dir", "")
  if not adapter_dir or not Path(adapter_dir).exists():
    raise FileNotFoundError(
        f"Set CFG['paths']['pretrained_adapter_dir'] to a valid adapter path. Got: {adapter_dir}")
  model = PeftModel.from_pretrained(model, adapter_dir)
  print(f"Loaded adapter for fast inference: {adapter_dir}")



In [ ]:
# ── 5d. Train (QLoRA) ─────────────────────────────────────────────────────
import json

train_df_run = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
val_df_run = val_df.copy()

if CFG["screening"]["max_train_samples"]:
  train_df_run = train_df_run.iloc[: int(
      CFG["screening"]["max_train_samples"])].reset_index(drop=True)
if CFG["screening"]["max_val_samples"]:
  val_df_run = val_df_run.iloc[: int(
      CFG["screening"]["max_val_samples"])].reset_index(drop=True)

print(f"Train samples (train+val): {len(train_df_run)} | Val split retained only for optional checks: {len(val_df_run)}")

train_ds_run = TrainScienceQADataset(train_df_run)
collator = TrainCollator(processor)

train_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    overwrite_output_dir=True,
    num_train_epochs=CFG["training"]["num_epochs"],
    per_device_train_batch_size=CFG["training"]["per_device_train_batch_size"],
    gradient_accumulation_steps=CFG["training"]["gradient_accumulation_steps"],
    learning_rate=CFG["training"]["learning_rate"],
    weight_decay=CFG["training"]["weight_decay"],
    warmup_ratio=CFG["training"]["warmup_ratio"],
    max_grad_norm=CFG["training"]["max_grad_norm"],
    logging_steps=CFG["training"]["logging_steps"],
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to=[],
    remove_unused_columns=False,
)

with open(RUN_DIR / "config.json", "w") as f:
  json.dump(CFG, f, indent=2)
print(f"Saved config: {RUN_DIR / 'config.json'}")

train_metrics = {}
if CFG["mode"]["do_train"]:
  trainer = Trainer(
      model=model,
      args=train_args,
      train_dataset=train_ds_run,
      data_collator=collator,
  )
  result = trainer.train()
  train_metrics = {k: float(v) if isinstance(
      v, (int, float)) else v for k, v in result.metrics.items()}
  model.save_pretrained(RUN_DIR / "best_adapter")
  processor.save_pretrained(RUN_DIR / "best_adapter")
  print(f"Saved adapter: {RUN_DIR / 'best_adapter'}")

with open(RUN_DIR / "train_metrics.json", "w") as f:
  json.dump(train_metrics, f, indent=2)
print(f"Saved train metrics: {RUN_DIR / 'train_metrics.json'}")


In [ ]:
# ── 5d2. Verbose strict evaluator helper ───────────────────────────────────
import time
import sys
from tqdm.auto import tqdm

def run_strict_eval_verbose(df: pd.DataFrame, has_label: bool, max_samples=None, tag="eval"):
    model.eval()
    rows = []
    correct = 0
    n_total = len(df) if max_samples is None else min(len(df), int(max_samples))
    t0 = time.time()
    pbar = tqdm(range(n_total), desc=f"Strict {tag}", mininterval=1.0, miniters=1)

    for i in pbar:
        row = df.iloc[i]
        pred_idx, valid_letters, valid_probs = score_choices_for_row(row)

        rec = {"id": row["id"], "num_choices": int(row["num_choices"]), "pred": pred_idx}
        if has_label:
            gt_idx = int(row["answer"])
            rec["answer"] = gt_idx
            rec["correct"] = int(pred_idx == gt_idx)
            correct += rec["correct"]

        for j, letter in enumerate(valid_letters):
            rec[f"p_{letter}"] = float(valid_probs[j])
        rows.append(rec)

        verbose_every = int(CFG.get("screening", {}).get("verbose_every", 10))
        if ((i + 1) % verbose_every == 0) or ((i + 1) == n_total):
            elapsed = time.time() - t0
            done = i + 1
            sps = done / max(elapsed, 1e-6)
            eta_sec = (n_total - done) / max(sps, 1e-6)
            msg = f"[{tag}] {done}/{n_total} | {sps:.3f} samples/s | elapsed={elapsed/60:.1f}m | eta={eta_sec/60:.1f}m"
            if has_label:
                msg += f" | running_acc={correct/done:.4f}"
            print(msg, flush=True)
            sys.stdout.flush()

    out_df = pd.DataFrame(rows)
    final_acc = (correct / len(out_df)) if (has_label and len(out_df) > 0) else None
    return out_df, final_acc

In [ ]:
# ── 5e. Strict val screening (and optional test submission) ───────────────
import json

if CFG["mode"]["do_val_eval"]:
  old_verbose = globals().get("VERBOSE_EVERY", 10)
  VERBOSE_EVERY = int(CFG["screening"]["verbose_every"])
  val_eval_df = val_df if CFG["screening"]["max_val_samples"] is None else val_df.iloc[:int(
      CFG["screening"]["max_val_samples"])].reset_index(drop=True)
  val_pred_df, val_acc = run_strict_eval_verbose(
      val_eval_df, has_label=True, max_samples=None, tag="val-screen")
  val_csv = RUN_DIR / "val_strict_eval_predictions.csv"
  val_pred_df.to_csv(val_csv, index=False)
  summary = {"val_acc": float(
      val_acc) if val_acc is not None else None, "n_val": int(len(val_eval_df))}
  with open(RUN_DIR / "val_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
  print(f"Validation accuracy: {val_acc:.4f}")
  print(f"Saved: {val_csv}")
  print(f"Saved: {RUN_DIR / 'val_summary.json'}")
  VERBOSE_EVERY = old_verbose

if CFG["mode"]["do_test_inference"]:
  test_pred_df, _ = run_strict_eval_verbose(
      test_df, has_label=False, max_samples=None, tag="test-screen")
  submission = pd.DataFrame(
      {"id": test_pred_df["id"], "answer": test_pred_df["pred"].astype(int)})
  sub_path = RUN_DIR / "submission.csv"
  submission.to_csv(sub_path, index=False)
  print(f"Saved submission: {sub_path}")